In [ ]:
!pip install -q dbt-duckdb

In [ ]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_name = vl.lakehouse_name
    dbt_path       = vl.dbt_path
    metadata_path  = vl.metadata_path
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
except ModuleNotFoundError:
    import yaml
    from pathlib import Path
    _root  = Path("C:/lakehouse/default/Files/dbt_fabric_python_notebook")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws_id"]
    lakehouse_id   = _cfg["lakehouse_id"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    metadata_path  = _cfg["metadata_path"]
os.environ['ROOT_PATH']           = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}'
os.environ['METADATA_LOCAL_PATH'] = metadata_path
os.environ['download_limit']      = download_limit
os.environ['process_limit']       = process_limit

In [ ]:
from dbt.cli.main import dbtRunner
os.chdir(dbt_path)
dbt = dbtRunner()
dbt.invoke(["run", "--target", "prod", "--profiles-dir", "."])
dbt.invoke(["test", "--target", "prod", "--profiles-dir", "."])